# 4. LLM-режим команды Graph-EVT-agent

**Цель:** запустить детерминированный численный `GraphEVTPipeline`, а затем передать его ограниченный evidence-пакет специализированным LLM-агентам из Graph-EVT-agent. LLM объясняет и проверяет результат, но не меняет порог, граф или ranking.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt ячейки ниже. Ключ не записывается в ноутбук или артефакты.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1

In [2]:
BASELINE_SIZE = int(0.75 * len(x))
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
# Численный результат доступен и воспроизводим независимо от LLM.
numeric_result = pipeline.run_detailed(x)
{"detected": numeric_result.detection.detected, "time_index": numeric_result.detection.time_index}

{'detected': False, 'time_index': None}

In [3]:
from getpass import getpass
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()


In [4]:
from getpass import getpass


def run_llm_review():
    api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
    if not api_key:
        print("LLM mode skipped: no MISTRAL_API_KEY was provided.")
        return None

    team = EVTAgentTeam(pipeline, MistralClient(api_key=api_key))
    return team.run(
        x,
        task="Проверь EVT-детекцию и ограничения вывода для одномерного синтетического ряда",
    )


agent_report = run_llm_review()
if agent_report is not None:
    print(agent_report.final_report)


HTTPError: HTTP Error 401: Unauthorized

`EVTAgentTeam` сначала вызывает тот же детерминированный pipeline. Агентам передаются агрегированные evidence-поля, а не право переписать вычисления. Для 1-D входа графовый и локализационный агенты корректно пропускаются. Текст LLM является интерпретацией и должен проверяться по `numeric_result`.